# Consumer Loan Risk — Scorecard Prototype

A transparent credit-risk screening prototype using a public Lending Club-derived extract.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score


In [2]:
df = pd.read_csv('../data/loans_full_schema.csv')
print(df.shape)
df.head()


(10000, 55)


,emp_title,emp_length,state,homeownership,annual_income,verified_income,debt_to_income,annual_income_joint,verification_income_joint,debt_to_income_joint,...,sub_grade,issue_month,loan_status,initial_listing_status,disbursement_method,balance,paid_total,paid_principal,paid_interest,paid_late_fees
0,global config engineer,3.0,NJ,MORTGAGE,90000.0,Verified,18.01,NaN,NaN,NaN,...,C3,Mar-2018,Current,whole,Cash,27015.86,1999.33,984.14,1015.19,0.0
1,warehouse office clerk,10.0,HI,RENT,40000.0,Not Verified,5.04,NaN,NaN,NaN,...,C1,Feb-2018,Current,whole,Cash,4651.37,499.12,348.63,150.49,0.0
2,assembly,3.0,WI,RENT,40000.0,Source Verified,21.15,NaN,NaN,NaN,...,D1,Feb-2018,Current,fractional,Cash,1824.63,281.80,175.37,106.43,0.0
3,customer service,1.0,PA,RENT,30000.0,Not Verified,10.16,NaN,NaN,NaN,...,A3,Jan-2018,Current,whole,Cash,18853.26,3312.89,2746.74,566.15,0.0
4,security supervisor,10.0,CA,RENT,35000.0,Verified,57.96,57000.0,Verified,37.66,...,C3,Mar-2018,Current,whole,Cash,21430.15,2324.65,1569.85,754.80,0.0


In [3]:
df['default_flag'] = df['loan_status'].isin(['Charged Off','Default']).astype(int)
df[['loan_status','default_flag']].head()


,loan_status,default_flag
0,Current,0
1,Current,0
2,Current,0
3,Current,0
4,Current,0


## DTI segmentation

In [4]:
df['dti_band'] = pd.cut(df['debt_to_income'], [-.01,10,20,30,40,100], labels=['<10%','10-20%','20-30%','30-40%','>40%'])
dti_summary = df.groupby('dti_band', observed=False)['default_flag'].agg(['count','mean']).rename(columns={'count':'applications','mean':'default_rate'})
dti_summary['default_rate_pct'] = dti_summary['default_rate'] * 100
dti_summary


,applications,default_rate,default_rate_pct
dti_band,,,
<10%,2121,0.000943,0.094295
10-20%,3788,0.000264,0.026399
20-30%,2709,0.001107,0.110742
30-40%,1020,0.000980,0.098039
>40%,305,0.000000,0.000000


## Logistic-regression scorecard prototype

In [5]:
features = ['debt_to_income','annual_income','loan_amount','interest_rate','grade','homeownership','verified_income','loan_purpose','term']
num = ['debt_to_income','annual_income','loan_amount','interest_rate']
cat = [c for c in features if c not in num]
X_train, X_test, y_train, y_test = train_test_split(df[features], df['default_flag'], test_size=.2, random_state=42, stratify=df['default_flag'])
preprocess = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num), ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat)])
model = Pipeline([('preprocess', preprocess), ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))])
model.fit(X_train, y_train)
prob = model.predict_proba(X_test)[:,1]
print('ROC-AUC:', round(roc_auc_score(y_test, prob), 3))
print('Default recall at 0.50:', round(recall_score(y_test, (prob >= .5).astype(int)), 3))


ROC-AUC: 0.102
Default recall at 0.50: 0.0


## Policy caution

The extract has only a small number of charged-off records. The score is therefore a portfolio triage prototype and must not be used as an automatic approval or rejection rule without a larger, time-split historical sample and calibration review.

## Note on statistical reliability

This extract contains only **7 charged-off loans out of 10,000** (a 0.07% observed rate). With that few positive cases, a stratified 20% test split leaves only 1-2 default events to score against, so the ROC-AUC and recall figures above are not statistically meaningful and should not be quoted as evidence the model works. The real, honest finding from this dataset is the base-rate scarcity itself: a production scorecard needs a materially larger historical default sample before any cut-off or AUC number can be trusted.